In [ ]:
import DiFfRG.phasediagram as pd
import DiFfRG.file_io as io
import DiFfRG.plot as plt
import numpy as np

In [ ]:
import glob
import os

import re

def remove_T_greater_than(names, x):
    return [
        name for name in names
        if not re.search(r"T:([0-9.]+)", name)
        or float(re.search(r"T:([0-9.]+)", name).group(1)) <= x
    ]

def remove_quadrature(names):
    return [name for name in names if "quadrature" not in name]

def load_data():
  global sims, csvs, params
  sims = glob.glob('./PhaseDiagram/*.log')
  sims = [sim[:-4].split('/')[-1] for sim in sims]
  sims = remove_quadrature(sims)
  sims = remove_T_greater_than(sims, 0.20)
  csvs = [f'./PhaseDiagram/{sim}_data_chiral.csv' for sim in sims]
  params = [io.get_parameters_from_name(sim) for sim in sims]
  
def clear_data():
  load_data()
  for i, csv in enumerate(csvs):
    if not os.path.exists(csv):
      baseName = './PhaseDiagram/' + sims[i]
      print('removing', baseName)
      os.system(f'rm -r {baseName}/ {baseName}.log {baseName}.log.json')
  load_data()
  
def tell_learner(learner):
  clear_data()
  for i, csv in enumerate(csvs):
    sigma = io.read_csv(csv)["sigmaGeV"].to_numpy()[-1]
    prm = params[i]
    print(i, " ", csv, " : ", sigma, " ", prm)
    learner.tell((prm['muq'], prm['T']), sigma)
  
def get_data():
  load_data()
  global in_data
  in_data = {}
  T = []
  muq = []
  sigma = []
  for i, csv in enumerate(csvs):
    csv_data = io.read_csv(csv)
    if(csv_data["t"].to_numpy()[-1] < 3.8):
      continue
    msigma = csv_data["sigmaGeV"].to_numpy()[-1]
    prm = params[i]
    T.append(prm['T'])
    muq.append(prm['muq'])
    sigma.append(msigma)
  in_data['T'] = np.array(T)
  in_data['muq'] = np.array(muq)
  in_data['sigma'] = np.array(sigma)

In [ ]:
import adaptive
adaptive.notebook_extension()
from concurrent.futures import ThreadPoolExecutor

folder = "./PhaseDiagram/"
parameter = "./parameter.json"
executable = "./build/QuarkMesonLPAprime"
observable_to_learn = "sigmaGeV"
loss_goal = 0.01

specification = [
    {"parameter": "/physical/muq", "bounds": [0.0, 0.4]},
    {"parameter": "/physical/T", "bounds": [0.0, 0.16]},
]

def solve_point(point):
    pd_point = [[spec["parameter"], point[i]] for i, spec in enumerate(specification)]
    name = pd.run_point(executable, pd_point, folder=folder, add_params=f"")
    csv = folder + name + "_data_chiral.csv" 
    ob = io.read_csv(csv)[observable_to_learn].to_numpy()[-1]
    return ob

bounds = []
for spec in specification:
    bounds.append(spec["bounds"])

learner = adaptive.LearnerND(solve_point, bounds=bounds)
tell_learner(learner)

executor = ThreadPoolExecutor(max_workers=8)
runner = adaptive.Runner(
    learner,
    executor=executor,
    loss_goal=loss_goal,
)
runner.live_info()  # shows a widget with status information
#runner.live_plot()

In [ ]:
import importlib
importlib.reload(plt)

get_data()

plt.plot_2D(
    {
        "x" : in_data['muq'] * 1e3,
        "y" : in_data['T'] * 1e3,
        "z" : in_data['sigma'] * 1e3,
    },
    cmap="rocket_r",
    levels=100,
    grid=True,
    xlabel=r"$\mu_q\,\textrm{[MeV]}$",
    ylabel=r"$T\,\textrm{[MeV]}$",
    zlabel=r"$\sigma\,\textrm{[MeV]}$",
    angle_3d=(25, 45),
    file="QM_PD.pdf"
)

In [ ]:
def print_adaptive_failures(runner):
    tbs = runner.tracebacks

    # Works for both older dict-style and newer list-of-tuples style
    if hasattr(tbs, "items"):
        items = tbs.items()
    else:
        items = tbs

    for point, tb in items:
        print("=" * 80)
        print("Failed point:", point)
        print("Last error line:", tb.strip().splitlines()[-1])
        print()
        print(tb)

print_adaptive_failures(runner)